# H3: does LoRA fine-tuning on diffusion counterfactuals transfer to real films?

Evaluation of the pre-registered H3 (PLAN.md, section 4, committed before any training): MedGemma 4B
trained on real films (R), RadEdit edits (E) or both (RE), 3 seeds each, against the base model.
Scores: float32 log-odds d of the trained question "Is there cardiomegaly in this image?" (P(yes) =
sigmoid(d)), from `python -m cxr.lora evaluate`; label: segmenter CTR > 0.5.

- Primary metrics: real-film CTR-AUROC (gain = arm - base; bar = 0.2 x (1 - base)); edited-film
  Brier for H3.1's first clause (gain = base - arm; bar = 0.2 x base). A gain "moves" the model if
  it reaches the bar with a CI lower bound above 0; it "does not move" it if its CI upper bound is
  below the bar.
- Statistics: each arm's metric is the mean over its seeds; 95% CIs from 1,000 paired bootstrap
  resamples of patients (the same patients for every model); a verdict also needs the deciding
  contrast to have the same sign for each seed k (seed k of every arm).
- The primary evaluation is the fresh test set (one final run of this committed notebook); the
  test split is secondary and exploratory. While some arms have no scores yet, they are replaced
  by the base model and the notebook says DRY RUN: that only checks the code path.

In [1]:
import sys
from pathlib import Path

import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent  # the notebook runs from notebooks/
R = ROOT / "results"
SET = "test"
ARMS, SEEDS = ["R", "E", "RE"], [0, 1, 2]
names = ["base"] + [f"{a}{s}" for a in ARMS for s in SEEDS] + [f"E{s}real" for s in SEEDS]
measures = pd.read_csv(R / f"h3_measures_{SET}.csv")
scores, scores_info, missing = {}, {}, []
for name in names:
    path = R / f"h3_scores_{name}_{SET}.csv"
    if path.exists():
        scores[name] = pd.read_csv(path)
        info = path.with_suffix(".json")
        scores_info[name] = json.loads(info.read_text()) if info.exists() else {}
    else:
        missing.append(name)
DRY_RUN = bool(missing)
for name in missing:  # dry run: a missing model is replaced by the base model (all gains 0)
    scores[name] = scores["base"]
print(f"Set: {SET}; models scored: {len(names) - len(missing)} of {len(names)}" + (f"; DRY RUN, replaced by base: {missing}" if DRY_RUN else ""))
revisions = {i.get("revision") for i in scores_info.values() if i.get("revision")}
assert len(revisions) <= 1, f"models scored at different MedGemma revisions: {revisions}"
for arm in ARMS:  # every seed of an arm trained on the same draw of examples
    draws = {scores_info[f"{arm}{s}"].get("train_draw") for s in SEEDS if f"{arm}{s}" not in missing}
    assert len(draws) <= 1, f"{arm}: seeds trained on different draws"
print("MedGemma revision:", revisions or "not recorded", "| code commits:", {i.get("commit") for i in scores_info.values()})
print(measures.groupby("kind").size().to_dict())

Set: test; models scored: 1 of 13; DRY RUN, replaced by base: ['R0', 'R1', 'R2', 'E0', 'E1', 'E2', 'RE0', 'RE1', 'RE2', 'E0real', 'E1real', 'E2real']
MedGemma revision: {'290cda5eeccbee130f987c4ad74a59ae6f196408'} | code commits: {'68e6d534b3c4341a14de73baf444b5bf3cece813'}
{'edit': 600, 'real': 1883, 'sham': 580}


## Per-image tables

In [2]:
keys = ["kind", "image", "growth"]
def table(name, question="cardiomegaly_0"):
    """One row per image for one model and question, with the measures (segmenter and CheXmask CTR,
    NIH labels): label = segmenter CTR > 0.5."""
    s = scores[name]
    s = s[s["question"] == question].merge(measures.drop(columns="patient_id"), on=keys, how="left")
    s["label"] = (s["ctr"] > 0.5).astype(float).where(s["ctr"].notna())
    return s.reset_index(drop=True)

tables = {name: table(name) for name in names}
base = tables["base"]
real_idx = base.index[(base["kind"] == "real") & base["ctr"].notna()]
edit_idx = base.index[(base["kind"] == "edit") & base["ctr"].notna()]
for name in names:  # every model is scored on the same images in the same order
    assert (tables[name][keys + ["patient_id"]].fillna(-1).values == base[keys + ["patient_id"]].fillna(-1).values).all(), name
print(f"real films with a CTR: {len(real_idx)} ({base.loc[real_idx, 'patient_id'].nunique()} patients); "
      f"edits with a CTR: {len(edit_idx)} ({base.loc[edit_idx, 'patient_id'].nunique()} patients)")

real films with a CTR: 1880 (974 patients); edits with a CTR: 600 (100 patients)


## The evaluation path reproduces the base values (test split only)

Pass rule, fixed before the evaluation path's first run. (a) Scoring path, which decides: the base
model's log-odds from `cxr.lora evaluate` against the dose run's (`results/dose_scores_4b.csv`), per
image; and the `results/h3_base.csv` metrics recomputed from the new log-odds with the dose run's
CTR labels. On the RTX PRO 6000 (the GPU of the re-score): log-odds identical except for at most 3
real films, metrics equal to 4 decimals; on another GPU: AUROCs within 0.002 and Brier scores within
0.001. (b) CTR path, reported on its own: agreement of CTR > 0.5 between the new measures and the
dose run's, expected at 99% or more.

In [3]:
old = pd.read_csv(R / "h3_base.csv")
old = old[(old["model"] == "4b") & (old["phrasing"].astype(str).isin(["0", "0.0"]))].set_index(["films", "metric"])["value"]
dm = pd.read_csv(R / "dose_measures.csv")
dose = pd.read_csv(R / "dose_scores_4b.csv").merge(dm.drop(columns="patient_id"), on=["group", "finding", "image", "level"])
dose = dose[(dose["phrasing"] == 0) & dose["group"].isin(["real", "dose"])].assign(
    kind=lambda x: np.where(x["group"] == "real", "real", "edit"), growth=lambda x: x["level"].where(x["group"] == "dose"))
both = base.merge(dose[["kind", "image", "growth", "log_odds", "ctr"]], on=keys, suffixes=("", "_dose"))
both = both[both["ctr_dose"].notna()]
diff = np.abs(both["log_odds"] - both["log_odds_dose"])
changed = diff > 1e-6  # float32 text read back as float64
print(f"(a) {len(both)} images: identical log-odds {1 - changed.mean():.4f}, changed {int(changed.sum())}, "
      f"largest change {diff.max():.4f}, Spearman {both['log_odds'].corr(both['log_odds_dose'], method='spearman'):.6f}")
both["y"] = (both["ctr_dose"] > 0.5).astype(int)
r, e = both[both["kind"] == "real"], both[both["kind"] == "edit"]
lab = r[(r["cardiomegaly"] == 1) | (r["no_finding"] == 1)]
new = {("real", "ctr_auroc"): roc_auc_score(r["y"], r["log_odds"]), ("real", "brier"): ((r["p_yes"] - r["y"]) ** 2).mean(),
       ("real", "nih_auroc"): roc_auc_score(lab["cardiomegaly"], lab["log_odds"]), ("real", "median_yes_no_mass"): r["yes_no_mass"].median(),
       ("edited", "ctr_auroc"): roc_auc_score(e["y"], e["log_odds"]), ("edited", "brier"): ((e["p_yes"] - e["y"]) ** 2).mean(),
       ("edited", "share_yes_below_05"): (e.loc[e["y"] == 0, "log_odds"] > 0).mean()}
check = pd.DataFrame([{"films": f, "metric": m, "h3_base": old[(f, m)], "evaluation_path": v, "difference": v - old[(f, m)]}
                      for (f, m), v in new.items()])
gpu_same = "RTX PRO 6000" in str(scores_info.get("base", {}).get("gpu", "RTX PRO 6000"))
if gpu_same:
    passed = int(changed.sum()) <= 3 and (check["difference"].abs() <= 5e-5 + 1e-12).all()  # h3_base.csv has 4 decimals
else:
    passed = all(abs(d) < (0.002 if "auroc" in m else 0.001) for m, d in zip(check["metric"], check["difference"]))
agree = (both["ctr"] > 0.5) == (both["ctr_dose"] > 0.5)
print(f"(b) CTR > 0.5 agreement with the dose run: {agree.mean():.4f}; disagreements: {int((~agree).sum())}, "
      f"largest |CTR - 0.5| among them: {(both.loc[~agree, 'ctr_dose'] - 0.5).abs().max() if (~agree).any() else 0:.4f}")
print("Reproduction:", "PASS" if passed else "FAIL")
assert passed, "the evaluation path does not reproduce h3_base.csv"
check.round(5)

(a) 2480 images: identical log-odds 0.9988, changed 3, largest change 0.1699, Spearman 0.999998
(b) CTR > 0.5 agreement with the dose run: 1.0000; disagreements: 0, largest |CTR - 0.5| among them: 0.0000
Reproduction: PASS


,films,metric,h3_base,evaluation_path,difference
0,real,ctr_auroc,0.8623,0.86225,-0.00005
1,real,brier,0.1889,0.18892,0.00002
2,real,nih_auroc,0.9202,0.92015,-0.00005
3,real,median_yes_no_mass,0.9855,0.98548,-0.00002
4,edited,ctr_auroc,0.9391,0.93911,0.00001
5,edited,brier,0.3187,0.31869,-0.00001
6,edited,share_yes_below_05,0.5518,0.55182,0.00002


## Metrics per model (point estimates)

In [4]:
prior = pd.read_csv(R / "h3_val_prior.csv").set_index("images")["share_ctr_above_05"]
pi_real, pi_edit = prior["real films"], prior["additions and shams"]  # val split (cxr.lora evaluate)
logit = lambda p: np.log(p / (1 - p))
print(f"prior correction: pi = {pi_real:.3f} (real films, also used for blanks), {pi_edit:.3f} (edits and shams); trained arms only")

def switch(d):
    """CTR where a logistic curve of P(yes) on CTR (P(yes) as a soft label) crosses 0.5, and its slope."""
    x = np.concatenate([d["ctr"], d["ctr"]])[:, None]
    y = np.concatenate([np.ones(len(d)), np.zeros(len(d))])
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(x, y, sample_weight=np.concatenate([d["p_yes"], 1 - d["p_yes"]]))
    return -lr.intercept_[0] / lr.coef_[0, 0], lr.coef_[0, 0]

def within_growth(d):
    num = den = 0.0
    for _, g in d.groupby("growth"):
        if 0 < g["label"].sum() < len(g):
            w = g["label"].sum() * (len(g) - g["label"].sum())
            num, den = num + w * roc_auc_score(g["label"], g["log_odds"]), den + w
    return num / den

def metrics(name):
    t, eff = tables[name], table(name, "effusion_0")
    r, e = t.loc[real_idx], t.loc[edit_idx]
    lab = r[(r["cardiomegaly"] == 1) | (r["no_finding"] == 1)]
    elab = eff[(eff["kind"] == "real") & eff["ctr"].notna() & ((eff["effusion"] == 1) | (eff["no_finding"] == 1))]  # real films = with a CTR
    sham, blank = t[t["kind"] == "sham"], t[t["kind"] == "blank"]
    q1 = table(name, "cardiomegaly_1").loc[real_idx]
    assert (q1[keys].fillna(-1).values == r[keys].fillna(-1).values).all(), "the two phrasings' rows are not aligned"
    out = {"model": name, "real_ctr_auroc": roc_auc_score(r["label"], r["log_odds"]),
           "real_brier": ((r["p_yes"] - r["label"]) ** 2).mean(),
           "real_brier_prior_corrected": np.nan if name == "base" else ((1 / (1 + np.exp(-(r["log_odds"] + logit(pi_real)))) - r["label"]) ** 2).mean(),
           "real_nih_auroc": roc_auc_score(lab["cardiomegaly"], lab["log_odds"]),
           "real_median_yes_no_mass": r["yes_no_mass"].median(),
           "edited_brier": ((e["p_yes"] - e["label"]) ** 2).mean(), "edited_ctr_auroc": roc_auc_score(e["label"], e["log_odds"]),
           "edited_within_growth_auroc": within_growth(e), "edited_yes_when_ctr_below_05": (e.loc[e["label"] == 0, "log_odds"] > 0).mean(),
           "sham_yes": (sham["log_odds"] > 0).mean(),
           "sham_yes_prior_corrected": np.nan if name == "base" else (sham["log_odds"] + logit(pi_edit) > 0).mean(),
           "blank_yes": (blank["log_odds"] > 0).mean(),
           "blank_yes_prior_corrected": np.nan if name == "base" else (blank["log_odds"] + logit(pi_real) > 0).mean(),
           "effusion_auroc": roc_auc_score(elab["effusion"], elab["log_odds"]),
           "transfer_phrasing_ctr_auroc": roc_auc_score(q1["label"], q1["log_odds"])}
    out["real_switch_ctr"], out["real_switch_slope"] = switch(r)
    if name != "base":  # also with the training prior removed, d + logit(pi): trained arms only (PLAN.md, H3)
        out["real_switch_ctr_prior_corrected"] = switch(r.assign(p_yes=1 / (1 + np.exp(-(r["log_odds"] + logit(pi_real))))))[0]
        d_edit = e["log_odds"] + logit(pi_edit)
        out["edited_brier_prior_corrected"] = ((1 / (1 + np.exp(-d_edit)) - e["label"]) ** 2).mean()
        out["edited_yes_when_ctr_below_05_prior_corrected"] = (d_edit[e["label"] == 0] > 0).mean()
    away = (r["ctr"] > 0.525) | (r["ctr"] < 0.475)  # also without the margin examples (as in training)
    out["real_ctr_auroc_without_margin"] = roc_auc_score(r.loc[away, "label"], r.loc[away, "log_odds"])
    for g, d in e.groupby("growth"):
        out[f"edited_median_p_yes_growth_{g:.2f}"] = d["p_yes"].median()
    return out

point = pd.DataFrame([metrics(n) for n in names]).set_index("model")
if not DRY_RUN:  # in a dry run the arms are copies of the base: no table goes to results/
    point.to_csv(R / f"h3_metrics_{SET}.csv")
point.round(3).T

prior correction: pi = 0.210 (real films, also used for blanks), 0.411 (edits and shams); trained arms only


model,base,R0,R1,R2,E0,E1,E2,RE0,RE1,RE2,E0real,E1real,E2real
real_ctr_auroc,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862,0.862
real_brier,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189,0.189
real_brier_prior_corrected,NaN,0.185,0.185,0.185,0.185,0.185,0.185,0.185,0.185,0.185,0.185,0.185,0.185
real_nih_auroc,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920,0.920
real_median_yes_no_mass,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985,0.985
edited_brier,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319,0.319
edited_ctr_auroc,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939,0.939
edited_within_growth_auroc,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873,0.873
edited_yes_when_ctr_below_05,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552,0.552
sham_yes,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195,0.195


## Paired patient bootstrap

1,000 resamples of patients, separately for the real films and for the edits' source patients, the
same for every model. Contrasts: R's, E's and RE's real-film CTR-AUROC gains, E's edited-film Brier
gain, D = R's real gain - 2 x E's real gain, RE - R (CTR-AUROC; secondary: Brier, NIH AUROC), and
the effusion AUROC drop of each arm (guard).

In [5]:
rng = np.random.default_rng(0)
real_pat = base.loc[real_idx, "patient_id"].to_numpy()
edit_pat = base.loc[edit_idx, "patient_id"].to_numpy()
groups_r = pd.Series(np.arange(len(real_idx))).groupby(real_pat).apply(list).tolist()
groups_e = pd.Series(np.arange(len(edit_idx))).groupby(edit_pat).apply(list).tolist()
D = {n: {"y_r": tables[n].loc[real_idx, "label"].to_numpy(), "d_r": tables[n].loc[real_idx, "log_odds"].to_numpy(),
         "p_r": tables[n].loc[real_idx, "p_yes"].to_numpy(), "y_e": tables[n].loc[edit_idx, "label"].to_numpy(),
         "p_e": tables[n].loc[edit_idx, "p_yes"].to_numpy()} for n in names}
nih = tables["base"].loc[real_idx, ["cardiomegaly", "no_finding", "effusion"]].to_numpy()
real_images = base.loc[real_idx, "image"].to_numpy()
for n in names:  # the effusion question, aligned with the real films
    D[n]["d_eff"] = table(n, "effusion_0").set_index("image").loc[real_images, "log_odds"].to_numpy()

def stats(ir, ie):
    """Per model: real CTR-AUROC, real Brier, real NIH AUROC, edited Brier and effusion AUROC
    (effusion vs no finding), on resampled indices."""
    res = {}
    keep = (nih[ir, 0] == 1) | (nih[ir, 1] == 1)
    keep_eff = (nih[ir, 2] == 1) | (nih[ir, 1] == 1)
    for n in names:
        x = D[n]
        res[n] = (roc_auc_score(x["y_r"][ir], x["d_r"][ir]), ((x["p_r"][ir] - x["y_r"][ir]) ** 2).mean(),
                  roc_auc_score(nih[ir, 0][keep], x["d_r"][ir][keep]), ((x["p_e"][ie] - x["y_e"][ie]) ** 2).mean(),
                  roc_auc_score(nih[ir, 2][keep_eff], x["d_eff"][ir][keep_eff]))
    return res

def contrasts(res, seeds=SEEDS):
    m = lambda arm, k: np.mean([res[f"{arm}{s}"][k] for s in seeds])
    b = res["base"]
    gain = {a: m(a, 0) - b[0] for a in ARMS}
    return {"gain_R_real": gain["R"], "gain_E_real": gain["E"], "gain_RE_real": gain["RE"],
            "gain_E_edited": b[3] - m("E", 3), "D": gain["R"] - 2 * gain["E"], "RE_minus_R_auroc": m("RE", 0) - m("R", 0),
            "R_minus_RE_brier": m("R", 1) - m("RE", 1), "RE_minus_R_nih_auroc": m("RE", 2) - m("R", 2),
            **{f"effusion_drop_{a}": b[4] - m(a, 4) for a in ARMS}}

full = contrasts(stats(np.arange(len(real_idx)), np.arange(len(edit_idx))))
boots = []
for _ in range(1000):
    ir = np.concatenate([groups_r[i] for i in rng.integers(len(groups_r), size=len(groups_r))])
    ie = np.concatenate([groups_e[i] for i in rng.integers(len(groups_e), size=len(groups_e))])
    boots.append(contrasts(stats(ir, ie)))
boots = pd.DataFrame(boots)
ci = pd.DataFrame({"point": full, "ci_low": boots.quantile(0.025), "ci_high": boots.quantile(0.975)})
per_seed = pd.DataFrame([contrasts(stats(np.arange(len(real_idx)), np.arange(len(edit_idx))), seeds=[k]) for k in SEEDS], index=SEEDS)
ci.round(4)

,point,ci_low,ci_high
gain_R_real,0.0,-0.0,0.0
gain_E_real,0.0,-0.0,0.0
gain_RE_real,0.0,-0.0,0.0
gain_E_edited,0.0,-0.0,0.0
D,0.0,-0.0,0.0
RE_minus_R_auroc,0.0,0.0,0.0
R_minus_RE_brier,0.0,0.0,0.0
RE_minus_R_nih_auroc,0.0,0.0,0.0
effusion_drop_R,0.0,-0.0,0.0
effusion_drop_E,0.0,-0.0,0.0


In [6]:
per_seed.round(4)

,gain_R_real,gain_E_real,gain_RE_real,gain_E_edited,D,RE_minus_R_auroc,R_minus_RE_brier,RE_minus_R_nih_auroc,effusion_drop_R,effusion_drop_E,effusion_drop_RE
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Robustness checks for H3.1 (point estimates of D)

In [7]:
def d_point(metric_fn):
    gain = lambda arm, real=False: np.mean([metric_fn(f"{arm}{s}" if not real else f"E{s}real") for s in SEEDS]) - metric_fn("base")
    return gain("R") - 2 * gain("E")

def auroc_with(label_col):
    def f(name):
        t = tables[name].loc[real_idx]
        t = t[t[label_col].notna()]
        return roc_auc_score(t[label_col] > 0.5, t["log_odds"])
    return f

def stratified(name):
    """CTR-AUROC over positive-negative pairs within NIH finding status only (no finding / any finding)."""
    t = tables[name].loc[real_idx]
    num = den = 0.0
    for _, g in t.groupby(t["no_finding"] == 1):
        if 0 < g["label"].sum() < len(g):
            w = g["label"].sum() * (len(g) - g["label"].sum())
            num, den = num + w * roc_auc_score(g["label"], g["log_odds"]), den + w
    return num / den

ctr_auroc = auroc_with("ctr")
real_gain_E_best_real = np.mean([ctr_auroc(f"E{s}real") for s in SEEDS]) - ctr_auroc("base")
robust = {"D with the CheXmask label": d_point(auroc_with("ctr_chexmask")),
          "D within NIH finding status": d_point(stratified),
          "D with E at its best real-validation checkpoint": full["gain_R_real"] - 2 * real_gain_E_best_real}
pd.Series(robust).round(4)

D with the CheXmask label                         -0.0
D within NIH finding status                        0.0
D with E at its best real-validation checkpoint    0.0
dtype: float64

## Verdicts (rules as pre-registered)

In [8]:
bar_real = 0.2 * (1 - point.loc["base", "real_ctr_auroc"])
bar_edit = 0.2 * point.loc["base", "edited_brier"]
c = ci.T
moves = lambda k, bar: c[k]["point"] >= bar and c[k]["ci_low"] > 0
does_not_move = lambda k, bar: c[k]["ci_high"] < bar
seeds_agree = lambda k, sign: bool((np.sign(per_seed[k]) == sign).all())
precondition = moves("gain_R_real", bar_real)
if not precondition:
    h31 = "inconclusive: R's real-film gain does not move the model (precondition)"
elif (moves("gain_E_edited", bar_edit) and seeds_agree("gain_E_edited", 1)  # both deciding contrasts: every seed
      and c["D"]["ci_low"] > 0 and seeds_agree("D", 1) and all(v > 0 for v in robust.values())):
    h31 = "supported"
else:
    ways = []  # both named ways are reported when both hold
    if does_not_move("gain_E_edited", bar_edit) and bool((per_seed["gain_E_edited"] < bar_edit).all()):
        ways.append("E did not learn the edits")
    if c["D"]["ci_high"] < 0 and seeds_agree("D", -1):
        ways.append("E transfers (more than half of R's real-film gain)")
    h31 = "falsified: " + "; ".join(ways) if ways else "inconclusive"
gain_ratio = c["gain_E_real"]["point"] / c["gain_R_real"]["point"] if c["gain_R_real"]["point"] != 0 else np.nan
diff = c["RE_minus_R_auroc"]
if diff["ci_low"] > 0 and diff["point"] >= 0.02 and seeds_agree("RE_minus_R_auroc", 1):
    h32 = "falsified: RE is better than R by at least 0.02 (RoentMod's prediction)"
elif not precondition:
    h32 = "inconclusive: precondition not met (only the falsification is evaluated)"
elif diff["ci_high"] < 0.02 and bool((per_seed["RE_minus_R_auroc"] < 0.02).all()):
    h32 = "supported: RE is no better than R"
else:
    h32 = "inconclusive"
guards = {f"{a}: blank yes (prior-corrected) > 5%": bool(np.mean([point.loc[f"{a}{s}", "blank_yes_prior_corrected"] for s in SEEDS]) > 0.05)
          for a in ARMS}
guards.update({f"{a}: effusion AUROC drop > 0.02, CI above 0": bool(c[f"effusion_drop_{a}"]["point"] > 0.02 and c[f"effusion_drop_{a}"]["ci_low"] > 0)
               for a in ARMS})
verdicts = pd.DataFrame([{"set": SET, "dry_run": DRY_RUN, "bar_real": bar_real, "bar_edited": bar_edit,
                          "precondition": precondition, "H3.1": h31, "H3.2": h32,
                          "ratio of real-film gains, E / R (description)": gain_ratio,
                          **{f"guard: {k}": v for k, v in guards.items()}, **{f"robustness: {k}": v for k, v in robust.items()}}])
if not DRY_RUN:
    verdicts.to_csv(R / f"h3_verdicts_{SET}.csv", index=False)
    ci.to_csv(R / f"h3_contrasts_{SET}.csv")
verdicts.T

,0
set,test
dry_run,True
bar_real,0.027549
bar_edited,0.063737
precondition,False
H3.1,inconclusive: R's real-film gain does not move...
H3.2,inconclusive: precondition not met (only the f...
"ratio of real-film gains, E / R (description)",NaN
guard: R: blank yes (prior-corrected) > 5%,False
guard: E: blank yes (prior-corrected) > 5%,False


## Takeaways

In [9]:
if DRY_RUN:
    print("DRY RUN: arms without scores were replaced by the base model; these verdicts only test the code path,"
          " and no metrics, contrasts or verdicts table was written to results/.")
print(f"Set {SET}: bars {bar_real:.3f} (real CTR-AUROC) and {bar_edit:.3f} (edited Brier).")
for k in ["gain_R_real", "gain_E_real", "gain_RE_real", "gain_E_edited", "D", "RE_minus_R_auroc"]:
    print(f"  {k}: {c[k]['point']:+.4f} (95% CI {c[k]['ci_low']:+.4f} to {c[k]['ci_high']:+.4f}); per seed {per_seed[k].round(4).tolist()}")
print(f"  ratio of real-film gains, E / R (description): {gain_ratio:.3f}")
print("H3.1:", h31)
print("H3.2:", h32)

DRY RUN: arms without scores were replaced by the base model; these verdicts only test the code path, and no metrics, contrasts or verdicts table was written to results/.
Set test: bars 0.028 (real CTR-AUROC) and 0.064 (edited Brier).
  gain_R_real: +0.0000 (95% CI -0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  gain_E_real: +0.0000 (95% CI -0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  gain_RE_real: +0.0000 (95% CI -0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  gain_E_edited: +0.0000 (95% CI -0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  D: +0.0000 (95% CI -0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  RE_minus_R_auroc: +0.0000 (95% CI +0.0000 to +0.0000); per seed [0.0, 0.0, 0.0]
  ratio of real-film gains, E / R (description): nan
H3.1: inconclusive: R's real-film gain does not move the model (precondition)
H3.2: inconclusive: precondition not met (only the falsification is evaluated)
